# Whisper Stage 1 Fine-Tuning on Colab - Tshivenda (ven)

Bootstrap notebook: run top to bottom on a fresh Colab GPU runtime. Counterpart to
`colab_stage1_wav2vec2_ven.ipynb` (Wav2Vec2) - same clone / data-regeneration / Drive-resume
pattern, swapped to Whisper large-v3 with the Seq2Seq training loop from
`src/asr/pilot_finetune_whisper_mps_ven.py`'s pilot recipe. This scales up the MPS pilot
result (WER 0.182 / CER 0.048 on a 12k-clip subset, 5 epochs - see
`notes/pilot-ven-results.md`) to the full ~60k-clip NCHLT+ANV pool.

**Before you start (one-time):**
1. Runtime -> Change runtime type -> **GPU**. whisper-large-v3 is large (~1.5B
   params) - an **A100** (Colab Pro) is strongly recommended. On a free-tier T4
   (16GB), keep the batch size small (the defaults below are tuned for a T4) or
   swap `MODEL` in step 6 to `openai/whisper-medium` if you hit OOM.
2. You need a Hugging Face account + access token (huggingface.co/settings/tokens, read scope).
3. The ANV/Swivuriso dataset is **gated**: while logged in on the HF website, open
   https://huggingface.co/datasets/dsfsi-anv/za-african-next-voices-compressed and accept
   the terms (approval is automatic). Without this, preprocessing fails on the ANV step.
4. This clones the `feature/tshivenda-classifier` branch - it must be pushed to GitHub first.

No language-token workaround needed here the way the CTC tokenizer needed a custom
`tokenizers/ven`: Whisper's BPE tokenizer already represents Tshivenda text fine.
The only stand-in is the placeholder language tag ("sw", Swahili - see
`src/asr/pilot_finetune_whisper_mps_ven.py`'s module docstring for why) used purely to give
the decoder a conditioning token.

## 1. GPU check

In [ ]:
# Should show a T4/V100/A100. If it errors: Runtime -> Change runtime type -> GPU
!nvidia-smi -L

## 2. Clone the repo + install dependencies

No macOS ffmpeg workaround needed here - Colab is Linux and torchcodec works with its system ffmpeg.

In [ ]:
!git clone -b feature/tshivenda-classifier https://github.com/Khotso-Bore/MultilingualASR.git
%cd MultilingualASR
# torch is preinstalled on Colab; this adds datasets/transformers/jiwer/etc.
!pip install -q -r requirements.txt

## 3. Hugging Face login (needed for the gated ANV dataset)

In [ ]:
# Paste your HF token when prompted (or store it as a Colab secret named HF_TOKEN
# and use userdata.get). The account must have accepted the ANV gate - see intro.
from huggingface_hub import login
login()

## 4. Mount Google Drive

Checkpoints and results go to Drive so a disconnected session can resume. Needs several GB free Drive space (checkpoints are pruned to the 2 most recent, but whisper-large-v3 checkpoints are ~3 GB each).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/multilingualasr"
OUTPUT_DIR = f"{DRIVE_ROOT}/whisper-ven-stage1"
PREDS_DIR = f"{DRIVE_ROOT}/preds-stage1-whisper"
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 5. Regenerate the processed data on this runtime

Downloads from HF and writes ~21 GB under `dataset/` (Colab disk, not Drive). NCHLT ~10 min, ANV ~30-60 min on Colab's connection. Re-runs are needed after every runtime reset - the wavs live on ephemeral disk (only checkpoints persist on Drive).

In [ ]:
!python src/preprocessing/preprocess_nchlt_ven.py
!python src/preprocessing/preprocess_anv_ven.py

## 6. Stage 1 training

Full NCHLT+ANV combined (~60k clips) - no 10s length cap like the MPS pilot needed
for memory safety. Whisper's feature extractor pads/truncates every clip to 30s
internally regardless, so long ANV clips are capped there rather than filtered out
up front. CUDA-adjusted from the pilot script (fp16 on, gradient checkpointing on).
**Resumes automatically** from the newest Drive checkpoint if the session died
mid-run - just re-run the notebook top to bottom.

Batch size 2 + grad accum 8 (effective batch 16) is sized for a T4's 16GB VRAM
against a 1.5B-parameter model - raise `per_device_train_batch_size` if you have
an A100.

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union
from datasets import load_dataset, Audio, Features, Value
from transformers import (
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    WhisperForConditionalGeneration, WhisperProcessor,
    EarlyStoppingCallback,
)
from transformers.trainer_utils import get_last_checkpoint
from jiwer import wer, cer

MODEL = "openai/whisper-large-v3"   # swap to openai/whisper-medium if you hit OOM on a T4
PLACEHOLDER_LANGUAGE = "sw"          # Swahili placeholder - see src/asr/pilot_finetune_whisper_mps_ven.py docstring

DATA = "dataset/processed"
features = Features({"audio": Audio(sampling_rate=16000), "transcript": Value("string")})
dataset_dict = load_dataset("csv", data_files={
    "train": [f"{DATA}/nchlt_ven/train.csv", f"{DATA}/anv_ven/train.csv"],
    "dev": [f"{DATA}/nchlt_ven/validation.csv", f"{DATA}/anv_ven/dev.csv"],
}, features=features)

processor = WhisperProcessor.from_pretrained(MODEL, language=PLACEHOLDER_LANGUAGE, task="transcribe")

def prepare_dataset(batch):
    samples = batch["audio"].get_all_samples()
    array = samples.data.numpy().squeeze()
    batch["input_features"] = processor.feature_extractor(array, sampling_rate=16000).input_features[0]
    batch["labels"] = processor.tokenizer(batch["transcript"]).input_ids
    return batch

dataset_dict = dataset_dict.map(prepare_dataset, remove_columns=dataset_dict["train"].column_names)

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, feats: List[Dict[str, Union[List[int], torch.Tensor]]]):
        input_features = [{"input_features": f["input_features"]} for f in feats]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in feats]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        # drop the forced BOS token if the tokenizer already prepended one -
        # the model re-adds it via forced_decoder_ids at generation time
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
    return {"wer": wer(label_str, pred_str), "cer": cer(label_str, pred_str)}

model = WhisperForConditionalGeneration.from_pretrained(MODEL)
model.generation_config.language = PLACEHOLDER_LANGUAGE
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,   # T4 16GB: raise to 8+ on an A100
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=225,
    logging_steps=50,
    learning_rate=1e-5,
    warmup_ratio=0.1,
    num_train_epochs=3,
    fp16=True,
    gradient_checkpointing=True,
    max_grad_norm=1.0,
    push_to_hub=False,
    report_to=[],
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_dict["train"],
    eval_dataset=dataset_dict["dev"],
    data_collator=DataCollatorSpeechSeq2SeqWithPadding(processor=processor),
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# resume from the newest Drive checkpoint if one exists (session died mid-run)
last_ckpt = get_last_checkpoint(OUTPUT_DIR)
print("resuming from:", last_ckpt or "scratch")
trainer.train(resume_from_checkpoint=last_ckpt)

trainer.save_model(f"{OUTPUT_DIR}/final")
processor.save_pretrained(f"{OUTPUT_DIR}/final")
print(f"saved -> {OUTPUT_DIR}/final")

## 7. Evaluate (comparable to the zero-shot baseline table)

Reuses `src/asr/zero_shot_baseline_ven.py` unchanged - it accepts any HF model id **or local
checkpoint path** via `--model`, so this scores the fine-tuned checkpoint on the same
held-out test sets (`nchlt_test`, `anv_dev_test`) with the same sampling and
normalisation as the zero-shot numbers already in `notes/pilot-ven-results.md`,
so the two are directly comparable. `--limit 0` scores the full test sets rather
than the 200-clip pilot sample. Predictions are saved to Drive - feed them to
`src/error_propagation/corrupt_transcripts_ven.py --error-model` for the error-propagation study.

In [ ]:
!python src/asr/zero_shot_baseline_ven.py --model {OUTPUT_DIR}/final --limit 0 --save-predictions {PREDS_DIR}